# LeetCode 1557 — Minimum Number of Vertices to Reach All Nodes bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-vertices-to-reach-all-nodes/ · Độ khó: **Medium** · Chủ đề: Graph Theory, Directed Acyclic Graph

Skeleton LeetCode:

```python
class Solution:
    def findSmallestSetOfVertices(self, n: int, edges: List[List[int]]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Biểu diễn **đồ thị có hướng** bằng danh sách cạnh và danh sách kề; hiểu **bậc vào (in-degree)** và **bậc ra (out-degree)**.
2. Viết baseline: thử các tập đỉnh theo kích thước tăng dần + BFS nhiều nguồn để kiểm tra phủ; và baseline "BFS từ mọi đỉnh".
3. Chứng minh được: trong **DAG**, tập cần tìm chính là **tập các đỉnh có bậc vào bằng 0**, và nó là duy nhất.
4. Cài đặt lời giải `O(n + E)` bằng mảng đếm bậc vào hoặc phép hiệu tập hợp.
5. Biết test bài "trả về theo thứ tự tuỳ ý" (so sau khi sort) và mở rộng sang đồ thị có chu trình (thành phần liên thông mạnh).

## 2. Đề bài & ví dụ

Cho một **đồ thị có hướng không chu trình** (DAG) gồm `n` đỉnh đánh số `0..n−1` và mảng `edges`, trong đó `edges[i] = [from_i, to_i]` là cạnh có hướng từ `from_i` tới `to_i`.

Tìm **tập đỉnh nhỏ nhất** sao cho từ các đỉnh trong tập có thể đi tới **mọi** đỉnh của đồ thị. Đề đảm bảo lời giải là duy nhất. Có thể trả về theo thứ tự bất kỳ.

**Ví dụ 1:** `n = 6`, `edges = [[0,1],[0,2],[2,5],[3,4],[4,2]]` → `[0,3]`. Từ `0` tới được `{0,1,2,5}`, từ `3` tới được `{3,4,2,5}`; một đỉnh duy nhất không đủ.

**Ví dụ 2:** `n = 5`, `edges = [[0,1],[2,1],[3,1],[1,4],[2,4]]` → `[0,2,3]`. Các đỉnh `0, 2, 3` không thể tới được từ đỉnh nào khác nên bắt buộc phải chọn; và chúng đi tới được `1`, `4`.

**Ràng buộc:** `2 ≤ n ≤ 10⁵`, `1 ≤ edges.length ≤ min(10⁵, n(n−1)/2)`, `edges[i].length == 2`, `0 ≤ from_i, to_i < n`, các cặp `(from_i, to_i)` đôi một khác nhau.

In [1]:
import random
import time
from collections import deque, defaultdict
from itertools import combinations
from typing import List

examples = [
    (6, [[0, 1], [0, 2], [2, 5], [3, 4], [4, 2]], [0, 3]),
    (5, [[0, 1], [2, 1], [3, 1], [1, 4], [2, 4]], [0, 2, 3]),
]
for n, edges, want in examples:
    print(f"n = {n}, edges = {edges} -> mong đợi {want} (thứ tự tuỳ ý)")

n = 6, edges = [[0, 1], [0, 2], [2, 5], [3, 4], [4, 2]] -> mong đợi [0, 3] (thứ tự tuỳ ý)
n = 5, edges = [[0, 1], [2, 1], [3, 1], [1, 4], [2, 4]] -> mong đợi [0, 2, 3] (thứ tự tuỳ ý)


## 3. Kiến thức nền

### 3.1 Danh sách kề (adjacency list)

Từ danh sách cạnh, dựng `adj[u]` = các đỉnh mà `u` có cạnh đi tới. Dùng `list` của `list`.

In [2]:
n, edges = examples[0][0], examples[0][1]
adj = [[] for _ in range(n)]
for u, v in edges:
    adj[u].append(v)
for u in range(n):
    print(u, "->", adj[u])

0 -> [1, 2]
1 -> []
2 -> [5]
3 -> [4]
4 -> [2]
5 -> []


### 3.2 Bậc vào / bậc ra

- **Bậc vào** `indeg[v]` = số cạnh đi **vào** `v`.
- **Bậc ra** `outdeg[u]` = số cạnh đi **ra** từ `u`.

Đỉnh có bậc vào 0 gọi là **nguồn (source)**, đỉnh có bậc ra 0 gọi là **đích/bồn (sink)**.

In [3]:
indeg = [0] * n
outdeg = [0] * n
for u, v in edges:
    outdeg[u] += 1
    indeg[v] += 1
print("indeg :", indeg)
print("outdeg:", outdeg)
print("nguồn :", [v for v in range(n) if indeg[v] == 0])

indeg : [0, 1, 2, 0, 1, 1]
outdeg: [2, 0, 1, 1, 1, 0]
nguồn : [0, 3]


### 3.3 BFS nhiều nguồn với `deque`

Đưa tất cả đỉnh xuất phát vào hàng đợi, đánh dấu đã thăm, rồi lan dần theo cạnh. Kết quả: tập mọi đỉnh tới được từ ít nhất một nguồn.

In [4]:
def reachable_from(n, adj, starts):
    seen = [False] * n
    q = deque()
    for s in starts:
        if not seen[s]:
            seen[s] = True
            q.append(s)
    while q:
        u = q.popleft()
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                q.append(v)
    return [v for v in range(n) if seen[v]]

print("từ {0}:", reachable_from(n, adj, [0]))
print("từ {3}:", reachable_from(n, adj, [3]))
print("từ {0,3}:", reachable_from(n, adj, [0, 3]))

từ {0}: [0, 1, 2, 5]
từ {3}: [2, 3, 4, 5]
từ {0,3}: [0, 1, 2, 3, 4, 5]


### 3.4 Phép hiệu tập hợp

`set(range(n)) - {v for _, v in edges}` = các đỉnh **không** là đích của cạnh nào = các đỉnh bậc vào 0. Phép hiệu `-` trên `set` chạy `O(len)`.

In [5]:
targets = {v for _, v in edges}
print("đích của cạnh:", targets)
print("không là đích:", sorted(set(range(n)) - targets))

đích của cạnh: {1, 2, 4, 5}
không là đích: [0, 3]


### 3.5 `itertools.combinations` để thử mọi tập con kích thước `k`

Dùng cho brute force: thử `k = 1, 2, ...`, gặp tập phủ đầu tiên thì đó là tập nhỏ nhất.

In [6]:
print(list(combinations(range(4), 2)))

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


## 4. Ý tưởng nền tảng / trực giác

Vẽ ví dụ 1 ra giấy:

```
0 ──► 1          3 ──► 4
│                      │
└──► 2 ◄───────────────┘
     │
     ▼
     5
```

Hỏi với từng đỉnh: "**có ai đi tới được tôi không?**"

| Đỉnh | Cạnh đi vào | Có thể tới từ đỉnh khác? |
|------|-------------|--------------------------|
| 0 | (không) | **không** ⇒ bắt buộc chọn |
| 1 | 0→1 | có |
| 2 | 0→2, 4→2 | có |
| 3 | (không) | **không** ⇒ bắt buộc chọn |
| 4 | 3→4 | có |
| 5 | 2→5 | có |

Đỉnh không có cạnh đi vào thì **chỉ** chính nó tới được nó ⇒ phải nằm trong đáp án. Đỉnh có cạnh đi vào thì "có người lo". Chọn `{0, 3}` và kiểm tra: đi tới được tất cả. Đáp án `[0, 3]`.

In [7]:
for n, edges, want in examples:
    has_in = [False] * n
    for u, v in edges:
        has_in[v] = True
    print(f"n={n}: đỉnh không có cạnh vào = {[v for v in range(n) if not has_in[v]]}, mong đợi {want}")

n=6: đỉnh không có cạnh vào = [0, 3], mong đợi [0, 3]
n=5: đỉnh không có cạnh vào = [0, 2, 3], mong đợi [0, 2, 3]


## 5. Cách 1 — Brute Force / baseline

### 5.1 Thử mọi tập con theo kích thước tăng dần

Với `k = 1, 2, ..., n`: duyệt mọi tập `k` đỉnh, BFS nhiều nguồn; tập đầu tiên phủ hết là đáp án (kích thước nhỏ nhất). Độ phức tạp `O(2ⁿ · (n + E))` — chỉ dùng với `n ≤ ~15`, làm chuẩn đối chiếu.

In [8]:
def brute_subsets(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    for k in range(1, n + 1):
        for S in combinations(range(n), k):
            if len(reachable_from(n, adj, S)) == n:
                return list(S)
    return list(range(n))

for n, edges, want in examples:
    print(brute_subsets(n, edges), "mong đợi", want)

[0, 3] mong đợi [0, 3]
[0, 2, 3] mong đợi [0, 2, 3]


### 5.2 BFS từ mọi đỉnh — "ai tới được ai"

Baseline đa thức: với mỗi đỉnh `u`, BFS tìm các đỉnh `u` tới được; đánh dấu các đỉnh đó (trừ `u`) là "được ai đó tới". Đáp án = các đỉnh **không** bị đánh dấu. Độ phức tạp `O(n · (n + E))` — với `n = 10⁵` là quá chậm.

In [9]:
def bfs_every_vertex(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    reached_by_other = [False] * n
    for u in range(n):
        for v in reachable_from(n, adj, [u]):
            if v != u:
                reached_by_other[v] = True
    return [v for v in range(n) if not reached_by_other[v]]

for n, edges, want in examples:
    print(bfs_every_vertex(n, edges), "mong đợi", want)

[0, 3] mong đợi [0, 3]
[0, 2, 3] mong đợi [0, 2, 3]


In [10]:
# Nút thắt: thời gian tăng theo n * (n + E)
def random_dag(n, e, rng):
    order = list(range(n))
    rng.shuffle(order)
    pairs = set()
    while len(pairs) < e:
        i, j = sorted(rng.sample(range(n), 2))
        pairs.add((order[i], order[j]))       # chỉ nối từ trước ra sau theo order -> không chu trình
    return [list(p) for p in pairs]

rng = random.Random(0)
for n in [200, 400, 800]:
    edges = random_dag(n, 2 * n, rng)
    t0 = time.perf_counter()
    bfs_every_vertex(n, edges)
    print(f"n={n}, E={2*n}: {time.perf_counter() - t0:.3f}s")

n=200, E=400: 0.002s
n=400, E=800: 0.007s
n=800, E=1600: 0.026s


**Nút thắt:** ta tính **toàn bộ** quan hệ tới được, trong khi chỉ cần biết "có cạnh nào đi vào đỉnh này không" — một thông tin cục bộ đọc thẳng từ danh sách cạnh.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (bắt buộc).** *Nếu* `indeg[v] = 0` *thì* không đỉnh nào khác tới được `v` ⇒ `v` **phải** có trong mọi tập phủ.

**Insight 2 (đủ).** *Nếu* `indeg[v] > 0` thì đi ngược một cạnh vào `v` tới `u₁`; nếu `indeg[u₁] > 0` lại đi ngược tiếp tới `u₂`, ... Vì đồ thị **không chu trình** nên không lặp lại đỉnh, dãy phải dừng sau tối đa `n` bước, và chỉ dừng được ở một đỉnh có bậc vào 0. Đỉnh đó tới được `v` theo chiều xuôi. ⇒ Tập các đỉnh bậc vào 0 **đã đủ** phủ mọi đỉnh.

**Kết luận:** đáp án = `{v : indeg[v] = 0}`. Nó vừa cần thiết (Insight 1) vừa đủ (Insight 2) ⇒ nhỏ nhất và **duy nhất** (giải thích vì sao đề nói "unique solution").

**Quan sát kỹ thuật:** ta không cần đếm chính xác bậc vào, chỉ cần biết `> 0` hay không ⇒ một mảng `bool` hoặc một `set` các đích là đủ.

In [11]:
# Minh hoạ Insight 2: đi ngược từ đỉnh 5 trong ví dụ 1 cho tới khi gặp nguồn
n, edges, _ = examples[0]
parents = defaultdict(list)
for u, v in edges:
    parents[v].append(u)
v = 5
path = [v]
while parents[v]:
    v = parents[v][0]
    path.append(v)
print("đi ngược:", " <- ".join(map(str, path)), "| dừng ở nguồn", v)

đi ngược: 5 <- 2 <- 0 | dừng ở nguồn 0


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đếm bậc vào bằng mảng

Từng đoạn:
1. `indeg = [0] * n`.
2. Duyệt cạnh `(u, v)`: `indeg[v] += 1`.
3. Trả `[v for v in range(n) if indeg[v] == 0]`.

In [12]:
def by_indegree(n, edges):
    indeg = [0] * n
    for _, v in edges:
        indeg[v] += 1
    return [v for v in range(n) if indeg[v] == 0]

for n, edges, want in examples:
    print(by_indegree(n, edges), "mong đợi", want)

[0, 3] mong đợi [0, 3]
[0, 2, 3] mong đợi [0, 2, 3]


Code nộp LeetCode cho cách 2:

In [13]:
class SolutionIndeg:
    def findSmallestSetOfVertices(self, n: int, edges: List[List[int]]) -> List[int]:
        indeg = [0] * n
        for _, v in edges:
            indeg[v] += 1
        return [v for v in range(n) if indeg[v] == 0]

print([SolutionIndeg().findSmallestSetOfVertices(n, e) for n, e, _ in examples])

[[0, 3], [0, 2, 3]]


**Độ phức tạp:** Time `O(n + E)`, Space `O(n)`.

### 7.2 Cách 3 — Mảng `bool` "có cạnh vào"

Giống cách 2 nhưng chỉ lưu `True/False` — thể hiện rõ ta chỉ cần biết "có hay không".

In [14]:
class SolutionBool:
    def findSmallestSetOfVertices(self, n: int, edges: List[List[int]]) -> List[int]:
        has_in = [False] * n
        for _, v in edges:
            has_in[v] = True
        return [v for v in range(n) if not has_in[v]]

print([SolutionBool().findSmallestSetOfVertices(n, e) for n, e, _ in examples])

[[0, 3], [0, 2, 3]]


### 7.3 Cách 4 — Hiệu tập hợp (một dòng)

`set(range(n)) - {v for _, v in edges}`. Trả về `list` (thứ tự tuỳ ý — được đề cho phép).

In [15]:
class SolutionSet:
    def findSmallestSetOfVertices(self, n: int, edges: List[List[int]]) -> List[int]:
        return list(set(range(n)) - {v for _, v in edges})

print([sorted(SolutionSet().findSmallestSetOfVertices(n, e)) for n, e, _ in examples])

[[0, 3], [0, 2, 3]]


**Độ phức tạp:** Time `O(n + E)` kỳ vọng (băm), Space `O(n)`. Ngắn nhất, nhưng thứ tự output không xác định — test phải so sau khi sort.

## 8. TRACE từng bước

Trace cách 2 trên ví dụ 2: `n = 5`, `edges = [[0,1],[2,1],[3,1],[1,4],[2,4]]`.

| Cạnh | indeg sau khi xử lý |
|------|---------------------|
| (bắt đầu) | [0, 0, 0, 0, 0] |
| 0→1 | [0, 1, 0, 0, 0] |
| 2→1 | [0, 2, 0, 0, 0] |
| 3→1 | [0, 3, 0, 0, 0] |
| 1→4 | [0, 3, 0, 0, 1] |
| 2→4 | [0, 3, 0, 0, 2] |

Các đỉnh có `indeg = 0`: `0, 2, 3` ⇒ đáp án `[0, 2, 3]`.

In [16]:
def trace_indegree(n, edges):
    indeg = [0] * n
    print("bắt đầu:", indeg)
    for u, v in edges:
        indeg[v] += 1
        print(f"  cạnh {u}->{v}: indeg[{v}] = {indeg[v]} | {indeg}")
    ans = []
    for v in range(n):
        tag = "CHỌN (bậc vào 0)" if indeg[v] == 0 else "bỏ qua"
        print(f"  đỉnh {v}: indeg = {indeg[v]} -> {tag}")
        if indeg[v] == 0:
            ans.append(v)
    print("đáp án:", ans)
    return ans

trace_indegree(5, [[0, 1], [2, 1], [3, 1], [1, 4], [2, 4]])

bắt đầu: [0, 0, 0, 0, 0]
  cạnh 0->1: indeg[1] = 1 | [0, 1, 0, 0, 0]
  cạnh 2->1: indeg[1] = 2 | [0, 2, 0, 0, 0]
  cạnh 3->1: indeg[1] = 3 | [0, 3, 0, 0, 0]
  cạnh 1->4: indeg[4] = 1 | [0, 3, 0, 0, 1]
  cạnh 2->4: indeg[4] = 2 | [0, 3, 0, 0, 2]
  đỉnh 0: indeg = 0 -> CHỌN (bậc vào 0)
  đỉnh 1: indeg = 3 -> bỏ qua
  đỉnh 2: indeg = 0 -> CHỌN (bậc vào 0)
  đỉnh 3: indeg = 0 -> CHỌN (bậc vào 0)
  đỉnh 4: indeg = 2 -> bỏ qua
đáp án: [0, 2, 3]


[0, 2, 3]

In [17]:
trace_indegree(6, [[0, 1], [0, 2], [2, 5], [3, 4], [4, 2]])

bắt đầu: [0, 0, 0, 0, 0, 0]
  cạnh 0->1: indeg[1] = 1 | [0, 1, 0, 0, 0, 0]
  cạnh 0->2: indeg[2] = 1 | [0, 1, 1, 0, 0, 0]
  cạnh 2->5: indeg[5] = 1 | [0, 1, 1, 0, 0, 1]
  cạnh 3->4: indeg[4] = 1 | [0, 1, 1, 0, 1, 1]
  cạnh 4->2: indeg[2] = 2 | [0, 1, 2, 0, 1, 1]
  đỉnh 0: indeg = 0 -> CHỌN (bậc vào 0)
  đỉnh 1: indeg = 1 -> bỏ qua
  đỉnh 2: indeg = 2 -> bỏ qua
  đỉnh 3: indeg = 0 -> CHỌN (bậc vào 0)
  đỉnh 4: indeg = 1 -> bỏ qua
  đỉnh 5: indeg = 1 -> bỏ qua
đáp án: [0, 3]


[0, 3]

## 9. Code nộp LeetCode

In [18]:
class Solution:
    def findSmallestSetOfVertices(self, n: int, edges: List[List[int]]) -> List[int]:
        # Trong DAG: tập cần tìm = các đỉnh không có cạnh đi vào
        has_in = [False] * n
        for _, v in edges:
            has_in[v] = True
        return [v for v in range(n) if not has_in[v]]

for n, edges, want in examples:
    got = Solution().findSmallestSetOfVertices(n, edges)
    print(got, want)
    assert sorted(got) == sorted(want)

[0, 3] [0, 3]
[0, 2, 3] [0, 2, 3]


## 10. Vì sao thuật toán đúng

Gọi `Z = {v : indeg[v] = 0}`.

1. **Mọi tập phủ đều chứa `Z`.** Với `v ∈ Z`, mọi đường đi tới `v` phải kết thúc bằng một cạnh vào `v` — không tồn tại. Vậy chỉ đường đi độ dài 0 (bắt đầu tại chính `v`) tới được `v` ⇒ `v` phải thuộc tập.
2. **`Z` là tập phủ.** Lấy `v ∉ Z`. Xây dãy `v = x₀ ← x₁ ← x₂ ← ...` với `x_{k+1} → x_k` là một cạnh, chừng nào `indeg[x_k] > 0`. Nếu dãy dài hơn `n` thì có đỉnh lặp ⇒ có chu trình, mâu thuẫn DAG. Vậy dãy dừng tại `x_t` có `indeg = 0`, tức `x_t ∈ Z`, và `x_t → ... → x₀ = v` là đường đi xuôi.
3. Từ (1) và (2): `Z` là tập phủ nhỏ nhất, và mọi tập phủ nhỏ nhất (cỡ `|Z|`, chứa `Z`) chính là `Z` ⇒ duy nhất.

**Lưu ý:** bước (2) cần tính không chu trình. Đồ thị có chu trình (ví dụ `0→1→0`) có thể có `Z = ∅` mà vẫn cần chọn ít nhất 1 đỉnh — xem bài tập 5.

In [19]:
# Kiểm chứng (1)+(2) thực nghiệm: Z phủ hết và brute force cho đúng Z trên DAG nhỏ
rng = random.Random(10)
for _ in range(300):
    n = rng.randint(2, 9)
    e = rng.randint(1, n * (n - 1) // 2)
    edges = random_dag(n, e, rng)
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    Z = by_indegree(n, edges)
    assert len(reachable_from(n, adj, Z)) == n
    assert sorted(brute_subsets(n, edges)) == Z
print("Z là tập phủ và trùng brute force: OK")

Z là tập phủ và trùng brute force: OK


## 11. Phân tích Time / Space

Gọi `n` = số đỉnh, `E` = số cạnh.

| Cách | Time | Space |
|------|------|-------|
| Thử tập con + BFS | `O(2ⁿ · (n + E))` | `O(n + E)` |
| BFS từ mọi đỉnh | `O(n · (n + E))` | `O(n + E)` |
| Đếm bậc vào / mảng bool | `O(n + E)` | `O(n)` |
| Hiệu tập hợp | `O(n + E)` kỳ vọng | `O(n)` |

Không cần dựng danh sách kề cho lời giải tối ưu ⇒ bộ nhớ chỉ `O(n)` (ngoài input).

In [20]:
rng = random.Random(1557)
n = 10**5
edges = random_dag(n, 10**5, rng)
for name, f in [("indeg", SolutionIndeg()), ("bool", SolutionBool()), ("set", SolutionSet())]:
    t0 = time.perf_counter()
    r = f.findSmallestSetOfVertices(n, edges)
    print(f"{name:5s}: |đáp án| = {len(r)} ({time.perf_counter() - t0:.3f}s)")

indeg: |đáp án| = 43123 (0.010s)
bool : |đáp án| = 43123 (0.008s)
set  : |đáp án| = 43123 (0.018s)


## 12. Bộ test

Vì thứ tự output tuỳ ý, mọi so sánh đều thực hiện sau khi `sorted`. Ngoài ra kiểm tra **tính hợp lệ**: tập trả về phủ hết đồ thị, và không có tập nhỏ hơn (qua brute force).

In [21]:
METHODS = {
    "brute_subsets": brute_subsets,
    "bfs_every_vertex": bfs_every_vertex,
    "by_indegree": by_indegree,
    "SolutionIndeg": lambda n, e: SolutionIndeg().findSmallestSetOfVertices(n, e),
    "SolutionBool": lambda n, e: SolutionBool().findSmallestSetOfVertices(n, e),
    "SolutionSet": lambda n, e: SolutionSet().findSmallestSetOfVertices(n, e),
    "Solution": lambda n, e: Solution().findSmallestSetOfVertices(n, e),
}
for n, edges, want in examples:
    for name, f in METHODS.items():
        assert sorted(f(n, edges)) == sorted(want), name
print("ví dụ đề: OK")

ví dụ đề: OK


In [22]:
# edge case
edge = [
    (2, [[0, 1]], [0]),                          # nhỏ nhất
    (3, [[0, 1]], [0, 2]),                        # đỉnh 2 cô lập (không xuất hiện trong edges)
    (4, [[0, 1], [1, 2], [2, 3]], [0]),           # đường thẳng
    (4, [[1, 0], [2, 0], [3, 0]], [1, 2, 3]),     # nhiều nguồn cùng chỉ vào 0
    (5, [[4, 3], [3, 2], [2, 1], [1, 0]], [4]),   # nguồn có chỉ số lớn nhất
    (4, [[0, 1], [0, 2], [0, 3], [1, 2], [1, 3], [2, 3]], [0]),   # DAG đầy đủ
]
for n, edges, want in edge:
    for name, f in METHODS.items():
        assert sorted(f(n, edges)) == want, (name, n, edges)
print("edge case: OK")

edge case: OK


In [23]:
# ngẫu nhiên có seed: DAG nhỏ, đối chiếu brute force + kiểm tra phủ
rng = random.Random(2023)
for _ in range(800):
    n = rng.randint(2, 9)
    e = rng.randint(1, n * (n - 1) // 2)
    edges = random_dag(n, e, rng)
    ref = sorted(brute_subsets(n, edges))
    for name, f in METHODS.items():
        assert sorted(f(n, edges)) == ref, (name, n, edges)
print("800 DAG ngẫu nhiên (seed 2023): mọi cách khớp brute force")

800 DAG ngẫu nhiên (seed 2023): mọi cách khớp brute force


In [24]:
# DAG lớn hơn: so các cách nhanh với BFS từ mọi đỉnh
rng = random.Random(99)
for _ in range(30):
    n = rng.randint(50, 200)
    e = rng.randint(1, 3 * n)
    edges = random_dag(n, e, rng)
    ref = bfs_every_vertex(n, edges)
    for name in ["by_indegree", "SolutionIndeg", "SolutionBool", "SolutionSet", "Solution"]:
        assert sorted(METHODS[name](n, edges)) == ref
print("30 DAG cỡ vừa: OK")

30 DAG cỡ vừa: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nhầm chiều: chọn đỉnh có **bậc ra** bằng 0

Đỉnh không có cạnh đi ra (sink) là nơi "kết thúc" — chọn chúng không giúp đi tới đâu cả.

In [25]:
def wrong_outdegree(n, edges):
    outdeg = [0] * n
    for u, _ in edges:
        outdeg[u] += 1                       # SAI: phải đếm bậc vào
    return [v for v in range(n) if outdeg[v] == 0]

for n, edges, want in examples:
    print(f"n={n} | sai: {wrong_outdegree(n, edges)} | đúng: {want}")

n=6 | sai: [1, 5] | đúng: [0, 3]
n=5 | sai: [4] | đúng: [0, 2, 3]


Sửa: đếm `indeg[v]` với `v` là **đích** của cạnh (`for _, v in edges`).

### Lỗi 2 — Chỉ xét các đỉnh xuất hiện trong `edges`

Đỉnh cô lập (không có cạnh nào) không xuất hiện trong `edges` nhưng vẫn phải được chọn.

In [26]:
def wrong_only_edge_vertices(n, edges):
    sources = {u for u, _ in edges}
    targets = {v for _, v in edges}
    return sorted(sources - targets)          # SAI: bỏ sót đỉnh cô lập

for n, edges in [(3, [[0, 1]]), (5, [[0, 1], [1, 2]])]:
    print(f"n={n}, edges={edges} | sai: {wrong_only_edge_vertices(n, edges)} | đúng: {Solution().findSmallestSetOfVertices(n, edges)}")

n=3, edges=[[0, 1]] | sai: [0] | đúng: [0, 2]
n=5, edges=[[0, 1], [1, 2]] | sai: [0] | đúng: [0, 3, 4]


Sửa: lấy hiệu từ `set(range(n))`, không phải từ tập các đỉnh nguồn của cạnh.

### Lỗi 3 — DFS theo thứ tự chỉ số: "chưa thăm thì chọn"

Duyệt `u = 0..n−1`, nếu `u` chưa được thăm thì cho vào đáp án và DFS đánh dấu. Sai khi một đỉnh chỉ số nhỏ lại **được tới** từ một đỉnh chỉ số lớn hơn.

In [27]:
def wrong_dfs_order(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    seen = [False] * n
    ans = []
    for u in range(n):
        if not seen[u]:
            ans.append(u)                    # SAI: u có thể được tới từ đỉnh chỉ số lớn hơn
            for x in reachable_from(n, adj, [u]):
                seen[x] = True
    return ans

for n, edges in [(2, [[1, 0]]), (5, [[4, 3], [3, 2], [2, 1], [1, 0]]), examples[0][:2]]:
    print(f"n={n} | sai: {wrong_dfs_order(n, edges)} | đúng: {Solution().findSmallestSetOfVertices(n, edges)}")

n=2 | sai: [0, 1] | đúng: [1]
n=5 | sai: [0, 1, 2, 3, 4] | đúng: [4]
n=6 | sai: [0, 3] | đúng: [0, 3]


Sửa: dùng tiêu chí bậc vào (không phụ thuộc thứ tự duyệt).

### Lỗi 4 — Dùng `list` để kiểm tra "là đích" (`v in targets_list`)

Kết quả đúng nhưng mỗi phép `in` trên `list` tốn `O(E)` ⇒ tổng `O(n·E)`, TLE với `n = E = 10⁵`. Ta đo trên cỡ nhỏ để thấy sự chênh lệch.

In [28]:
def slow_list_lookup(n, edges):
    targets = [v for _, v in edges]               # CHẬM: list
    return [v for v in range(n) if v not in targets]

rng = random.Random(5)
n = 4000
edges = random_dag(n, 4000, rng)
t0 = time.perf_counter(); r1 = slow_list_lookup(n, edges); t1 = time.perf_counter()
r2 = Solution().findSmallestSetOfVertices(n, edges); t2 = time.perf_counter()
print(f"list: {t1 - t0:.3f}s | mảng bool: {t2 - t1:.4f}s | cùng kết quả: {r1 == r2}")

list: 0.100s | mảng bool: 0.0004s | cùng kết quả: True


Sửa: dùng mảng `bool`/đếm hoặc `set`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có **DAG** + "tới được mọi đỉnh" / "ít nguồn nhất" ⇒ nghĩ ngay tới **bậc vào = 0**.
- Đề nói "**unique solution**" là gợi ý mạnh rằng đáp án có cấu trúc xác định (ở đây: tập nguồn).
- Hỏi "**đỉnh này có thể tới từ đỉnh khác không?**" — thường chỉ cần thông tin cục bộ (cạnh vào), không cần BFS.
- Output thứ tự tuỳ ý ⇒ test bằng `sorted` hoặc so `set`.
- Bậc vào còn là nền tảng của **topological sort (Kahn)**, phát hiện chu trình (Course Schedule, LC 207).

## 15. Biến thể của bài

1. **Đồ thị có chu trình**: nén thành phần liên thông mạnh (SCC) thành DAG, đáp án = số SCC có bậc vào 0 (mỗi SCC chọn 1 đỉnh). Xem bài tập 5.
2. **Đồ thị vô hướng**: đáp án = số thành phần liên thông (mỗi thành phần chọn 1 đỉnh).
3. **Thêm ít cạnh nhất để từ một đỉnh tới được tất cả** (Reachability from capital, Codeforces 999E) — đếm SCC nguồn không chứa thủ đô.
4. **Thêm ít cạnh nhất để đồ thị liên thông mạnh**: `max(#nguồn, #bồn)` của DAG nén (trừ trường hợp 1 SCC).
5. **Find the Town Judge (LC 997)**: cũng đếm bậc vào / bậc ra.

In [29]:
# Biến thể 2: đồ thị vô hướng -> số thành phần liên thông (mỗi thành phần chọn 1 đỉnh)
def undirected_min_vertices(n, edges):
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for u, v in edges:
        parent[find(u)] = find(v)
    return len({find(v) for v in range(n)})

print(undirected_min_vertices(6, [[0, 1], [0, 2], [2, 5], [3, 4], [4, 2]]), undirected_min_vertices(4, [[0, 1]]))

1 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1a. Tập con + BFS | thử mọi tập theo kích thước | `O(2ⁿ(n+E))` | `O(n+E)` | kiểm chứng, `n ≤ 15` |
| 1b. BFS mọi đỉnh | ai được ai tới | `O(n(n+E))` | `O(n+E)` | đồ thị nhỏ, cần cả quan hệ tới được |
| 2. Đếm bậc vào | nguồn = bậc vào 0 | `O(n+E)` | `O(n)` | **bản nộp**, kèm số liệu bậc |
| 3. Mảng bool | chỉ cần "có cạnh vào?" | `O(n+E)` | `O(n)` | **bản nộp** gọn |
| 4. Hiệu tập hợp | `range(n) − đích` | `O(n+E)` | `O(n)` | một dòng, thứ tự tuỳ ý |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** Kết quả (đã sort) của `n = 4, edges = [[0,1],[1,2],[2,3]]` và `n = 4, edges = [[1,0],[2,0],[3,0]]`?

**Bài 2 (tính tay):** Với `n = 7`, `edges = [[0,3],[1,3],[3,4],[2,5],[5,4]]` (đỉnh 6 không có cạnh nào). Lập bảng bậc vào và cho đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_indegree(n, edges)` trả về mảng bậc vào.

**Bài 4 (tự viết lại):** Viết `ex4_is_cover(n, edges, S)` trả `True` nếu từ tập `S` đi tới được mọi đỉnh (BFS nhiều nguồn).

**Bài 5 (mini challenge):** Viết `ex5_min_vertices_any(n, edges)` cho đồ thị có hướng **có thể có chu trình** (không khuyên, cạnh khác nhau): trả về **một** tập nhỏ nhất phủ mọi đỉnh.

In [30]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_indegree(n, edges):
    indeg = [0] * n
    for u, v in edges:
        # TODO: cạnh u -> v làm tăng bậc vào của đỉnh nào?
        pass
    return indeg

In [31]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_is_cover(n, edges, S):
    # TODO
    raise NotImplementedError

def ex5_min_vertices_any(n, edges):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tìm đỉnh không có cạnh vào. (2) Đường `0→1→2→3`: chỉ `0`. (3) `[0]` và `[1, 2, 3]`.

**Bài 2.** (1) `indeg = [0,0,0,2,2,1,0]`. (2) Đỉnh 6 cô lập vẫn có bậc vào 0. (3) Đáp án `[0, 1, 2, 6]`.

**Bài 3.** (1) Cạnh `u → v` đi **vào** `v`. (2) `indeg[v] += 1`. (3) `u` không dùng tới.

**Bài 4.** (1) Dựng `adj`. (2) Đưa mọi đỉnh của `S` vào hàng đợi, đánh dấu. (3) So số đỉnh đã thăm với `n`.

**Bài 5.** (1) Đỉnh `u`, `v` cùng SCC ⇔ tới được nhau. (2) Một SCC "nguồn" (không có cạnh từ SCC khác vào) phải đóng góp ≥ 1 đỉnh; chọn đúng 1 đỉnh mỗi SCC nguồn là đủ (DAG nén). (3) Với `n` nhỏ: tính tập tới được của mọi đỉnh; `v` thuộc SCC nguồn ⇔ mọi `u` tới được `v` thì `v` cũng tới được `u`; chọn đỉnh chỉ số nhỏ nhất của mỗi SCC nguồn. Kiểm chứng bằng `brute_subsets` (so **kích thước** và tính hợp lệ, vì đáp án không duy nhất).

## 19. Đáp án tham khảo cho bài tập

In [32]:
# Bài 1
a1 = (sorted(Solution().findSmallestSetOfVertices(4, [[0, 1], [1, 2], [2, 3]])),
      sorted(Solution().findSmallestSetOfVertices(4, [[1, 0], [2, 0], [3, 0]])))
print("Bài 1:", a1)
assert a1 == ([0], [1, 2, 3])

# Bài 2
a2 = trace_indegree(7, [[0, 3], [1, 3], [3, 4], [2, 5], [5, 4]])
assert a2 == [0, 1, 2, 6] == brute_subsets(7, [[0, 3], [1, 3], [3, 4], [2, 5], [5, 4]])

Bài 1: ([0], [1, 2, 3])
bắt đầu: [0, 0, 0, 0, 0, 0, 0]
  cạnh 0->3: indeg[3] = 1 | [0, 0, 0, 1, 0, 0, 0]
  cạnh 1->3: indeg[3] = 2 | [0, 0, 0, 2, 0, 0, 0]
  cạnh 3->4: indeg[4] = 1 | [0, 0, 0, 2, 1, 0, 0]
  cạnh 2->5: indeg[5] = 1 | [0, 0, 0, 2, 1, 1, 0]
  cạnh 5->4: indeg[4] = 2 | [0, 0, 0, 2, 2, 1, 0]
  đỉnh 0: indeg = 0 -> CHỌN (bậc vào 0)
  đỉnh 1: indeg = 0 -> CHỌN (bậc vào 0)
  đỉnh 2: indeg = 0 -> CHỌN (bậc vào 0)
  đỉnh 3: indeg = 2 -> bỏ qua
  đỉnh 4: indeg = 2 -> bỏ qua
  đỉnh 5: indeg = 1 -> bỏ qua
  đỉnh 6: indeg = 0 -> CHỌN (bậc vào 0)
đáp án: [0, 1, 2, 6]


In [33]:
# Bài 3
def ex3_indegree_sol(n, edges):
    indeg = [0] * n
    for u, v in edges:
        indeg[v] += 1
    return indeg

assert ex3_indegree_sol(5, [[0, 1], [2, 1], [3, 1], [1, 4], [2, 4]]) == [0, 3, 0, 0, 2]
assert ex3_indegree_sol(3, [[0, 1]]) == [0, 1, 0]
print("Bài 3: OK")

Bài 3: OK


In [34]:
# Bài 4
def ex4_is_cover_sol(n, edges, S):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    return len(reachable_from(n, adj, list(S))) == n

n, edges, _ = examples[0]
assert ex4_is_cover_sol(n, edges, [0, 3]) is True
assert ex4_is_cover_sol(n, edges, [0]) is False
assert ex4_is_cover_sol(n, edges, [3, 1, 0]) is True
rng4 = random.Random(4)
for _ in range(300):
    n = rng4.randint(2, 10)
    edges = random_dag(n, rng4.randint(1, n * (n - 1) // 2), rng4)
    assert ex4_is_cover_sol(n, edges, Solution().findSmallestSetOfVertices(n, edges))
print("Bài 4: OK")

Bài 4: OK


In [35]:
# Bài 5 — đồ thị có thể có chu trình
def ex5_min_vertices_any_sol(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    reach = [set(reachable_from(n, adj, [u])) for u in range(n)]
    ans = []
    for v in range(n):
        # v thuộc SCC nguồn: ai tới được v thì v cũng tới được người đó
        if all(u in reach[v] for u in range(n) if v in reach[u]):
            # chỉ lấy đại diện nhỏ nhất của SCC
            if all(not (w < v and w in reach[v] and v in reach[w]) for w in range(n)):
                ans.append(v)
    return ans

def random_digraph(n, e, rng):
    pairs = set()
    while len(pairs) < e:
        u, v = rng.sample(range(n), 2)
        pairs.add((u, v))
    return [list(p) for p in pairs]

assert sorted(ex5_min_vertices_any_sol(3, [[0, 1], [1, 0], [2, 0]])) == [2]
assert len(ex5_min_vertices_any_sol(2, [[0, 1], [1, 0]])) == 1
rng5 = random.Random(5)
for _ in range(400):
    n = rng5.randint(2, 8)
    edges = random_digraph(n, rng5.randint(1, n * (n - 1)), rng5)
    got = ex5_min_vertices_any_sol(n, edges)
    assert ex4_is_cover_sol(n, edges, got)
    assert len(got) == len(brute_subsets(n, edges)), (n, edges)
# trên DAG, kết quả phải trùng tập nguồn
for _ in range(200):
    n = rng5.randint(2, 9)
    edges = random_dag(n, rng5.randint(1, n * (n - 1) // 2), rng5)
    assert ex5_min_vertices_any_sol(n, edges) == by_indegree(n, edges)
print("Bài 5: OK")

Bài 5: OK

## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def findSmallestSetOfVertices(self, n, edges):
        has_in = [False] * n
        for _, v in edges:
            has_in[v] = True
        return [v for v in range(n) if not has_in[v]]
```

**Template "nguồn của DAG":** đếm bậc vào → các đỉnh bậc vào 0. Đồ thị có chu trình → nén SCC trước.

**Checklist trước khi submit:**
- [ ] Đếm **bậc vào** (đích của cạnh), không phải bậc ra.
- [ ] Duyệt **mọi** đỉnh `0..n−1` (đỉnh cô lập vẫn được chọn).
- [ ] Dùng mảng/`set`, không `in` trên `list`.
- [ ] Nhớ lý do đúng: DAG ⇒ đi ngược luôn dừng ở một nguồn.
- [ ] Thử nhanh: ví dụ 1 → `[0,3]`, ví dụ 2 → `[0,2,3]`, `n=3, [[0,1]] → [0,2]`.